In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.Newton(
    model=model,
    mu=1e-3,
    lr_init=1e-3,
    damping="identity",
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.6199186444282532
epoch:  1, loss: 0.6105325222015381
epoch:  2, loss: 0.601132869720459
epoch:  3, loss: 0.5917254090309143
epoch:  4, loss: 0.5823737382888794
epoch:  5, loss: 0.5730588436126709
epoch:  6, loss: 0.5638168454170227
epoch:  7, loss: 0.5546808838844299
epoch:  8, loss: 0.5456854104995728
epoch:  9, loss: 0.536827027797699
epoch:  10, loss: 0.5280935168266296
epoch:  11, loss: 0.5194843411445618
epoch:  12, loss: 0.5110164880752563
epoch:  13, loss: 0.5026847124099731
epoch:  14, loss: 0.4944842457771301
epoch:  15, loss: 0.4863949716091156
epoch:  16, loss: 0.478435218334198
epoch:  17, loss: 0.4705863296985626
epoch:  18, loss: 0.46284472942352295
epoch:  19, loss: 0.45519983768463135
epoch:  20, loss: 0.44765469431877136
epoch:  21, loss: 0.44020941853523254
epoch:  22, loss: 0.4328629672527313
epoch:  23, loss: 0.42560890316963196
epoch:  24, loss: 0.4184527099132538
epoch:  25, loss: 0.4113931357860565
epoch:  26, loss: 0.4044323265552521
epoch:  2

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.951971709728241
Test metrics:  R2 = 0.9011400938034058


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    mu=1e-3,
    damping=True,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.05840878188610077
epoch:  1, loss: 0.05129795894026756
epoch:  2, loss: 0.045478228479623795
epoch:  3, loss: 0.03940531983971596
epoch:  4, loss: 0.034984298050403595
epoch:  5, loss: 0.031174391508102417
epoch:  6, loss: 0.017919475212693214
epoch:  7, loss: 0.016252627596259117
epoch:  8, loss: 0.015923235565423965
epoch:  9, loss: 0.006531450431793928
epoch:  10, loss: 0.0052143787033855915
epoch:  11, loss: 0.0041690850630402565
epoch:  12, loss: 0.0034523229114711285
epoch:  13, loss: 0.0029927357099950314
epoch:  14, loss: 0.0026422804221510887
epoch:  15, loss: 0.0023846407420933247
epoch:  16, loss: 0.0021911405492573977
epoch:  17, loss: 0.0020455399062484503
epoch:  18, loss: 0.0019378077704459429
epoch:  19, loss: 0.0018257995834574103
epoch:  20, loss: 0.0017399777425453067
epoch:  21, loss: 0.0016678348183631897
epoch:  22, loss: 0.001596895162947476
epoch:  23, loss: 0.0015351037727668881
epoch:  24, loss: 0.0014867502031847835
epoch:  25, loss: 0.0014

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9873690009117126
Test metrics:  R2 = 0.8275281190872192
